In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

agg = pd.read_csv("../data/processed/messages_clean.csv")
agg['first_date'] = pd.to_datetime(agg['first_date'])

top = agg.sort_values('n_occurrences', ascending=False).head(5)
for _, r in top.iterrows():
    print(r['n_occurrences'], r['n_groups'], r['text'][:150].replace("\n", " "))
    print("---")

2248 63 Acesse este link para entrar no meu grupo do WhatsApp: https://chat.whatsapp.com/Lx27UrUZZdv3hANbOM9IjN
---
124 1 BEM-VINDOS(AS)  ✅ POR FAVOR AO ENTRAR SE APRESENTAR COM FOTO,NOME E IDADE VOCÊS TERÁ 5 MINUTOS PRA SE APRESENTAR SEJA ATIVO OU SERA REMOVIDO  PROIBIDO
---
67 2 Não manda link dessa bosta no meu grupo não seu corno
---
67 12 😷😷😷😷😷😷 Abra este link para entrar no meu Grupo no WhatsApp: https://chat.whatsapp.com/Ixz01PQUVSl4aj4Z1dpZwM
---
63 1 ╔═══════════╗ ╔═╝███████████╚═╗ ╔╝███████████████╚╗ ║█████████████████║ ║█████████████████║ ║█████████████████║ ║█╔█████████████╗█║ ╚╦╝███▒▒███▒▒███╚╦
---


In [2]:
def cluster_from_vectors(X, threshold, k=10):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute").fit(X)
    dist, idx = nn.kneighbors(X)
    rows, cols = [], []
    for i in range(X.shape[0]):
        for d, j in zip(dist[i], idx[i]):
            if i != j and (1 - d) >= threshold:
                rows.append(i)
                cols.append(j)
    n = X.shape[0]
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n))
    _, labels = connected_components(g, directed=False)
    return labels

def summarize(labels, name):
    s = pd.Series(labels).value_counts()
    multi = s[s >= 2]
    print(f"[{name}] clusters with 2+ messages: {len(multi)}, "
          f"messages inside them: {multi.sum()}, largest cluster: {multi.max()}")

In [3]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(agg['clean'])

for th in [0.5, 0.6, 0.7]:
    labels = cluster_from_vectors(X_tfidf, th)
    summarize(labels, f"tfidf th={th}")

[tfidf th=0.5] clusters with 2+ messages: 983, messages inside them: 3167, largest cluster: 154
[tfidf th=0.6] clusters with 2+ messages: 887, messages inside them: 2653, largest cluster: 39
[tfidf th=0.7] clusters with 2+ messages: 806, messages inside them: 2222, largest cluster: 21


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb = model.encode(agg['clean'].tolist(), batch_size=64,
                   show_progress_bar=True, normalize_embeddings=True)
np.save("../data/processed/emb_minilm.npy", emb)
print(emb.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/316 [00:00<?, ?it/s]

(20176, 384)


In [5]:
for th in [0.80, 0.85, 0.90]:
    labels = cluster_from_vectors(emb, th)
    summarize(labels, f"minilm th={th}")

[minilm th=0.8] clusters with 2+ messages: 944, messages inside them: 5374, largest cluster: 2083
[minilm th=0.85] clusters with 2+ messages: 806, messages inside them: 3031, largest cluster: 561
[minilm th=0.9] clusters with 2+ messages: 619, messages inside them: 1940, largest cluster: 200


In [7]:
# if the kernel restarted, reload first: emb = np.load("../data/processed/emb_minilm.npy")
labels_m = cluster_from_vectors(emb, 0.85)
labels_t = cluster_from_vectors(X_tfidf, 0.6)

def peek(labels, name, n_clusters=3, n_samples=4):
    s = pd.Series(labels).value_counts()
    for cid in s.index[:n_clusters]:
        members = agg[labels == cid]
        print(f"=== {name} | cluster {cid} | size {len(members)}")
        for t in members['text'].sample(min(n_samples, len(members)), random_state=0):
            print("  -", t[:120].replace("\n", " "))

peek(labels_m, "minilm 0.85")
peek(labels_t, "tfidf 0.6")

=== minilm 0.85 | cluster 9 | size 561
  - *ᴀǫᴜɪ ᴛᴇʀᴇᴍᴏѕ:*  _🔥➛ ᴄᴏɴᴠᴇʀѕᴀ_ _🔥➛ ɢɪғ  ᴘᴏʀɴᴏ_ _🔥➛ ɪᴍᴀɢᴇᴍ ᴘᴏʀɴᴏ_ _🔥➛ ғɪɢᴜʀɪɴʜᴀ ᴘᴏʀɴᴏ_ _🔥➛ ғɪɢᴜʀɪɴʜᴀ ɴᴏʀᴍᴀʟ_  *ғᴏᴄᴏ ᴅᴏ ɢʀ
  - *• ✧🎀───🤍 ✾ ───🤍 ✧🎀•*         *✧ѕ̫a̫r̫a̫н̫ѕ̫т̫a̫т̫υ̫ѕ̫ o̫ғ̫c̫✧*          *✧ѕaraн✧ ғraѕeѕ✧*  - *тalvez eυ não тenнo aѕ мe
  - *./)_/)* *( ́。•᎑•)*  *∪🎀∪──🍒D͎E͎U͎S͎🦋──────╮* *┊ꕤ᎒ꦿ⃟ۜ♱⸙Δ̸₦Ĵ〄 ϜȒΔ̸₷玄₷⸙♱ ⃟ꦿ⸼*  *┊ ⚈ ꒳⚈)  ⌲=͟͟͞͞A=͟͟͞͞N=͟͟͞͞J=͟͟͞͞O  🛸* *ᅠ╭
  - https://chat.whatsapp.com/Fz2x7TBJcYFKxE8n75Ilyq  ⛱💜༻╍╍⊰✾⊱* *👉🏻ąʠųį ƥǫdę ƥǫşţąŗ*👈🏻  *꙳ ꦿ⃟ۜ𖣔💜⊰✾⊱ ɠįƒş*🎑 *꙳ ꦿ⃟ۜ𖣔⛱⊰✾⊱ Wallp
=== minilm 0.85 | cluster 630 | size 92
  - Abra este link para entrar no meu Grupo no WhatsApp: https://chat.whatsapp.com/LXNBoonotwiCqScq1sAwnH  ༺✮•°◤seja todos b
  - Acesse este link para entrar no meu grupo do WhatsApp: https://chat.whatsapp.com/BYyZX7dRYzr42GAnni3oCf  regras basicas🤗
  - Acesse este link para entrar no meu grupo do WhatsApp: https://chat.whatsapp.com/Fz3hzO6b5YrEDrJdcVBTQj🍓 🍓 FRASES, LINkS
  - Acesse este link para entrar no meu grupo 

In [8]:
spam_re = r"chat\.whatsapp\.com|bem[- ]vind|link.{0,40}grupo|grupo.{0,40}link|entrar no (meu )?grupo"
is_spam = agg['text'].str.contains(spam_re, case=False, regex=True)

def alpha_ratio(t):
    return sum(c.isalpha() for c in t) / max(len(t), 1)
is_art = agg['text'].apply(alpha_ratio) < 0.5

keep = ~(is_spam | is_art)
print("removed spam:", is_spam.sum(), " removed art:", is_art.sum(), " kept:", keep.sum())

agg_f = agg[keep].reset_index(drop=True)
emb_f = emb[keep.values]
agg_f.to_csv("../data/processed/messages_filtered.csv", index=False)
np.save("../data/processed/emb_filtered.npy", emb_f)

# mutual-neighbour clustering: an edge needs both messages to list each other
def cluster_mutual(X, threshold, k=10):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute").fit(X)
    dist, idx = nn.kneighbors(X)
    rows, cols = [], []
    for i in range(X.shape[0]):
        for d, j in zip(dist[i], idx[i]):
            if i != j and (1 - d) >= threshold:
                rows.append(i); cols.append(j)
    n = X.shape[0]
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n)).tocsr()
    g = g.minimum(g.T)
    _, labels = connected_components(g, directed=False)
    return labels

tfidf_f = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_tfidf_f = tfidf_f.fit_transform(agg_f['clean'])

for th in [0.5, 0.6, 0.7]:
    summarize(cluster_mutual(X_tfidf_f, th), f"tfidf mutual th={th}")
for th in [0.80, 0.85, 0.90]:
    summarize(cluster_mutual(emb_f, th), f"minilm mutual th={th}")

/tmp/ipykernel_16083/4050456418.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_spam = agg['text'].str.contains(spam_re, case=False, regex=True)


removed spam: 766  removed art: 1898  kept: 17672
[tfidf mutual th=0.5] clusters with 2+ messages: 716, messages inside them: 1881, largest cluster: 15
[tfidf mutual th=0.6] clusters with 2+ messages: 604, messages inside them: 1588, largest cluster: 15
[tfidf mutual th=0.7] clusters with 2+ messages: 545, messages inside them: 1415, largest cluster: 15
[minilm mutual th=0.8] clusters with 2+ messages: 853, messages inside them: 3730, largest cluster: 858
[minilm mutual th=0.85] clusters with 2+ messages: 646, messages inside them: 1955, largest cluster: 133
[minilm mutual th=0.9] clusters with 2+ messages: 487, messages inside them: 1277, largest cluster: 17


In [9]:
labels_t = cluster_mutual(X_tfidf_f, 0.6)
labels_m = cluster_mutual(emb_f, 0.90)

agg_f['cl_tfidf'] = labels_t
agg_f['cl_minilm'] = labels_m
agg_f['size_tfidf'] = pd.Series(labels_t).map(pd.Series(labels_t).value_counts()).values
agg_f['size_minilm'] = pd.Series(labels_m).map(pd.Series(labels_m).value_counts()).values

def peek_random(col, size_col, name, n_clusters=6, n_samples=3, seed=1):
    multi = agg_f[agg_f[size_col] >= 2]
    ids = pd.Series(multi[col].unique()).sample(n_clusters, random_state=seed)
    for cid in ids:
        members = agg_f[agg_f[col] == cid]
        print(f"=== {name} | cluster {cid} | size {len(members)}")
        for t in members['text'].head(n_samples):
            print("  -", t[:200].replace("\n", " "))

peek_random('cl_tfidf', 'size_tfidf', 'tfidf 0.6')
peek_random('cl_minilm', 'size_minilm', 'minilm 0.90')

agg_f.to_csv("../data/processed/messages_clustered.csv", index=False)

=== tfidf 0.6 | cluster 7138 | size 3
  - Globo News Marcelo Cosme: Faça  o que eu digo, mas não faça o que eu faço,. 🤥🤣🤣🤣🤣🤣
  - 👆🏼Marcelo Cosme da Globonews em pauta. "Faça o que eu digo, mas não faça o que eu faço"
  - Repórter da Globo News.  Faça o que eu mando, não faça o que eu faço
=== tfidf 0.6 | cluster 8602 | size 2
  - Marinha Americana nas águas do mar do Caribe, rumo a VENEZUELA  Difíceis dias... *socialistas* só param de fazer merda quando são tirados do poder a força ou qdo. morrem. O óbvio seria Nicolas Maduro 
  - Marinha Americana nas águas do mar do Caribe, rumo a VENEZUELA  Difíceis dias... *socialistas* só param de fazer merda quando são tirados do poder a força ou qdo. morrem. O óbvio seria Nicolas Maduro 
=== tfidf 0.6 | cluster 10716 | size 3
  - *O que está por trás de Mandetta x Bolsonaro*  Luiz Henrique Mandetta é um político, filho de político e de família de políticos. É caboclo manhoso, não prega prego sem estopa.  Andou magoado porque q
  - *O que está po

In [10]:
def export_eval(col, size_col, name, n=50, seed=42):
    multi = agg_f[agg_f[size_col] >= 2]
    ids = pd.Series(multi[col].unique())
    ids = ids.sample(min(n, len(ids)), random_state=seed)
    rows = []
    for cid in ids:
        m = agg_f[agg_f[col] == cid]
        rows.append({
            "method": name,
            "cluster_id": cid,
            "size": len(m),
            "samples": " ||| ".join(t[:250].replace("\n", " ") for t in m['text'].head(4)),
            "same_family (1/0)": ""
        })
    return pd.DataFrame(rows)

ev = pd.concat([export_eval('cl_tfidf', 'size_tfidf', 'tfidf'),
                export_eval('cl_minilm', 'size_minilm', 'minilm')])
ev = ev.sample(frac=1, random_state=0).reset_index(drop=True)  # shuffled so you can't tell which method
ev.to_csv("../outputs/manual_eval.csv", index=False)
print(len(ev))

100


In [11]:
ev = pd.read_csv("../outputs/manual_eval.csv")
ev[['method', 'cluster_id']].to_csv("../outputs/eval_key.csv", index=False)
ev.drop(columns=['method', 'cluster_id']).to_csv("../outputs/manual_eval_blind.csv", index=False)
print(len(ev))

100


In [17]:
try:
    print("model already loaded:", type(model).__name__)
except NameError:
    print("model NOT loaded")

model already loaded: SentenceTransformer


In [20]:
import random, re
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.cluster import pair_confusion_matrix

try:
    model
except NameError:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

rng = random.Random(0)
FOOTER = "compartilhe com todos os seus contatos agora mesmo"
HEADER = "atenção urgente leia até o final"

def drop(frac):
    def f(t):
        w = [x for x in t.split() if rng.random() > frac]
        return " ".join(w) if len(w) >= 5 else None
    return f

def swaps(t):
    w = t.split()
    for i in range(len(w) - 1):
        if rng.random() < 0.15:
            w[i], w[i + 1] = w[i + 1], w[i]
    return " ".join(w)

def numbers(t):
    if not re.search(r"\d", t):
        return None
    return re.sub(r"\d+", lambda m: str(int(m.group()) + rng.randint(1, 50)), t)

MUT = {
    "numbers_changed": numbers,
    "footer_added": lambda t: t + " " + FOOTER,
    "header_added": lambda t: HEADER + " " + t,
    "drop_15pct": drop(0.15),
    "drop_30pct": drop(0.30),
    "truncate_60pct": lambda t: " ".join(t.split()[:int(len(t.split()) * 0.6)]),
    "word_swaps": swaps,
}

# sources: real messages that were NOT in any natural cluster, so no hidden duplicates
cand = agg_f[(agg_f['size_tfidf'] == 1) & (agg_f['size_minilm'] == 1)
             & (agg_f['clean'].str.split().str.len() >= 15)]
src_ids = cand.sample(400, random_state=0).index.tolist()

rows = []
for s in src_ids:
    base = agg_f.loc[s, 'clean']
    for name, fn in MUT.items():
        v = fn(base)
        if v is not None and v != base:
            rows.append({"src": s, "mut": name, "text": v})
synth = pd.DataFrame(rows)
print(len(src_ids), "sources ->", len(synth), "planted variants")
print(synth['mut'].value_counts())

400 sources -> 2497 planted variants
mut
footer_added       400
header_added       400
drop_30pct         400
truncate_60pct     400
word_swaps         388
drop_15pct         382
numbers_changed    127
Name: count, dtype: int64


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

n = len(agg_f)
emb_synth = model.encode(synth['text'].tolist(), batch_size=64,
                         show_progress_bar=True, normalize_embeddings=True)
emb_all = np.vstack([emb_f, emb_synth])

texts_all = agg_f['clean'].tolist() + synth['text'].tolist()
tf_all = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_all = tf_all.fit_transform(texts_all)
print(emb_all.shape, X_all.shape)

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

(20169, 384) (20169, 172237)


In [22]:
src_pos = np.array(src_ids)
var_pos = n + np.arange(len(synth))
planted = np.concatenate([src_pos, var_pos])
truth = np.concatenate([src_pos, synth['src'].values])

configs = [("tfidf", t) for t in (0.5, 0.6, 0.7)] + [("minilm", t) for t in (0.85, 0.90, 0.95)]
res = []
for name, th in configs:
    X = X_all if name == "tfidf" else emb_all
    labels = cluster_mutual(X, th)
    pred = labels[planted]
    C = pair_confusion_matrix(truth, pred)
    tp, fp, fn = C[1, 1], C[0, 1], C[1, 0]
    row = {"method": name, "th": th,
           "precision": round(tp / (tp + fp), 3),
           "recall": round(tp / (tp + fn), 3),
           "ARI": round(adjusted_rand_score(truth, pred), 3)}
    same = labels[var_pos] == labels[synth['src'].values]
    for m in MUT:
        row[m] = round(same[(synth['mut'] == m).values].mean(), 2)
    res.append(row)

results = pd.DataFrame(res)
pd.set_option("display.width", 250)
print(results.to_string(index=False))
results.to_csv("../outputs/synthetic_eval.csv", index=False)

method   th  precision  recall   ARI  numbers_changed  footer_added  header_added  drop_15pct  drop_30pct  truncate_60pct  word_swaps
 tfidf 0.50      1.000   0.992 0.996             1.00          1.00          1.00        1.00        0.97            1.00        1.00
 tfidf 0.60      1.000   0.960 0.979             1.00          1.00          1.00        0.99        0.86            1.00        0.99
 tfidf 0.70      1.000   0.857 0.923             1.00          1.00          1.00        0.96        0.61            0.96        0.92
minilm 0.85      0.953   0.841 0.893             0.98          0.99          0.98        0.95        0.74            0.72        1.00
minilm 0.90      1.000   0.691 0.817             0.95          0.96          0.88        0.87        0.55            0.48        1.00
minilm 0.95      1.000   0.404 0.575             0.85          0.82          0.50        0.56        0.21            0.24        0.97


In [23]:
import difflib

def cohesion(col, size_col):
    fam = agg_f[agg_f[size_col] >= 2]
    sims = []
    for cid, g in fam.groupby(col):
        t = g.sort_values('first_date')['clean'].tolist()
        for x in t[1:]:
            sims.append(difflib.SequenceMatcher(None, t[0].split(), x.split(), autojunk=False).ratio())
    s = pd.Series(sims)
    return dict(pairs=len(s), mean_sim=round(s.mean(), 3),
                share_below_0_3=round((s < 0.3).mean(), 3))

print("tfidf 0.6 :", cohesion('cl_tfidf', 'size_tfidf'))
print("minilm 0.90:", cohesion('cl_minilm', 'size_minilm'))

tfidf 0.6 : {'pairs': 984, 'mean_sim': np.float64(0.796), 'share_below_0_3': np.float64(0.022)}
minilm 0.90: {'pairs': 790, 'mean_sim': np.float64(0.795), 'share_below_0_3': np.float64(0.067)}


In [24]:
print(results[['method', 'th', 'word_swaps']].to_string(index=False))

method   th  word_swaps
 tfidf 0.50        1.00
 tfidf 0.60        0.99
 tfidf 0.70        0.92
minilm 0.85        1.00
minilm 0.90        1.00
minilm 0.95        0.97
